# Long short-term memory (LSTM) model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re, string, warnings, random
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize

from nltk.corpus import stopwords
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

import nlpaug.augmenter.word as naw

tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

In [1]:
df = pd.read_csv('reviews.csv')
print(f"Shape: {df.shape}")
df.head(10)

NameError: name 'pd' is not defined

In [ ]:
df.info()

## Text Preprocessing

- **Lowercasing:** Reduces vocabulary size 
- **Remove URLs / emails / special characters:** These carry no sentiment and just add noise
- **Remove extra whitespace:** Normalises the input
- **Stop words:** We intentionally keep some stop words like "not", "no", "very" because they carry sentiment.

In [ ]:
df = pd.read_csv('reviews.csv')
print(f"Shape: {df.shape}")

stop_words = set(stopwords.words('english'))
sentiment_keepers = {
    'not', 'no', 'nor', 'never', 'neither', 'nobody', 'nothing', 'nowhere',
    'very', 'too', 'most', 'more', 'less', 'least', 'few',
    'but', 'however', 'although', 'only', 'just',
    'above', 'below', 'again', 'against',
}
stop_words = stop_words - sentiment_keepers

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = ' '.join(w for w in text.split() if w not in stop_words)
    return text

df['clean_text'] = df['review_text'].apply(clean_text)
df = df[df['clean_text'].str.len() > 0].reset_index(drop=True)

print("\nClass distribution before augmentation:")
print(df['rating'].value_counts().sort_index())

### Class imbalance

The dataset is heavily imbalanced and rating 5 makes up almost half of the data, while ratings 2 and 3
have very few samples. This could be a problem because the model could simply learn to predict 5 for 
every input, achieving almost 50% accuracy without actually learning anything

**What we did to handle this:**
- **Class weights** during training so we penalise mistakes on minority classes more heavily.
- **Synonym replacement** to generate more data for the underrepresented classes.
- **Macro-averaged F1** as the main metric. This treats all classes equally regardless of size.

## Split Off Test Set from the Original Data

We reserve 15% of the original, unaugmented data as our test set.
This guarantees the test set contains only real reviews and no synthetic samples.
Augmentation only happens on the remaining 85% (train + val).

In [ ]:
df_trainval, df_test = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df['rating']
)
df_trainval = df_trainval.reset_index(drop=True)
df_test     = df_test.reset_index(drop=True)

print(f"Train+Val set: {len(df_trainval)} samples  (will be augmented)")
print(f"Test set:      {len(df_test)} samples  (original data only)")

## Text Augmentation on Minority Classes

We use synonym replacement from `nlpaug` to generate new, slightly varied versions of reviews
in the underrepresented classes (ratings 2, 3, 4).

The target is to bring every minority class up to the same count as **rating 1** (the second largest class),
so we don't aggressively oversample all the way up to rating 5.

Each augmented sample replaces ~20% of words with synonyms with enough variation to avoid
pure duplication, while keeping the original meaning and sentiment

In [ ]:
aug = naw.SynonymAug(aug_src='wordnet', aug_p=0.2)

count_per_class = df_trainval['rating'].value_counts().sort_index()
target_count = count_per_class[1] 
print(f"Target count per class: {target_count}")

augmented_rows = []

for rating in [2, 3, 4]:
    class_df = df_trainval[df_trainval['rating'] == rating]
    current_count = len(class_df)
    needed = target_count - current_count
    print(f"Rating {rating}: {current_count} samples, generating {needed} augmented samples")

    samples = class_df['clean_text'].tolist()
    sources = random.choices(samples, k=needed)
    augmented_texts = aug.augment(sources)

    for text in augmented_texts:
        augmented_rows.append({'review_text': text, 'clean_text': text, 'rating': rating})

aug_df = pd.DataFrame(augmented_rows)
df_trainval_balanced = pd.concat([df_trainval, aug_df], ignore_index=True)
df_trainval_balanced = df_trainval_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nTrain+Val distribution after augmentation:")
print(df_trainval_balanced['rating'].value_counts().sort_index())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, data, title in zip(axes,
                            [df_trainval, df_trainval_balanced],
                            ['Train+Val: Before Augmentation', 'Train+Val: After Augmentation']):
    pct = data['rating'].value_counts(normalize=True).sort_index() * 100
    pct.plot(kind='bar', ax=ax, color=sns.color_palette('viridis', 5))
    ax.set_title(title)
    ax.set_ylabel('Percentage')
    ax.set_xlabel('Rating')
    ax.set_ylim(0, 55)
plt.tight_layout()
plt.show()

## Tokenisation & Padding

We convert each review into a sequence of integer tokens and pad/truncate to a fixed length
so that every input to the LSTM has the same shape.

**Parameters chosen here:**
- MAX_VOCAB = 20 000 Keeps the most common words because rare words add more noise 
- MAX_LEN = 100 Covers most of the reviews without wasting memory

## Prepare Labels & Split Data

The rating column has values 1–5. We subtract 1 so labels become 0–4 which is required by
`to_categorical`. We then split into **train (70 %)**, **validation (15 %)** (used to check early stopping), and **test (15 %)**

In [ ]:
MAX_VOCAB   = 20_000
MAX_LEN     = 100
NUM_CLASSES = 5

# Fit tokenizer on augmented trainval only to avoid leakage
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(df_trainval_balanced['clean_text'])

def tokenize_and_pad(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')

X_trainval = tokenize_and_pad(df_trainval_balanced['clean_text'])
y_trainval  = df_trainval_balanced['rating'].values - 1

X_test_final = tokenize_and_pad(df_test['clean_text'])
y_test_final  = df_test['rating'].values - 1

print(f"Train+Val shape: {X_trainval.shape}")
print(f"Test shape:      {X_test_final.shape}")

## Split Train+Val into Train (82%) and Val (18%)

~70/15/15 overall: 15% test is already set aside, so split remaining 85% into 82/18

In [ ]:
X_train, X_val, y_train_int, y_val_int = train_test_split(
    X_trainval, y_trainval, test_size=0.18, random_state=42, stratify=y_trainval
)

y_train = to_categorical(y_train_int, NUM_CLASSES)
y_val   = to_categorical(y_val_int,   NUM_CLASSES)

print(f"Train: {X_train.shape[0]}  Val: {X_val.shape[0]}  Test: {X_test_final.shape[0]}")

## Compute class weights

Because the dataset is imbalanced, we compute class weights that are inversely proportional
to class frequency. This is done by Scikit-learn's `compute_class_weight('balanced', ...)`

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=y_train_int
)
class_weight_dict = dict(enumerate(class_weights))

print("Class weights:")
for cls, weight in class_weight_dict.items():
    print(f"  Rating {cls + 1}: {weight:.3f}")

## Building the model

- `Embedding` Learns a vector for each word
- `SpatialDropout1D(0.3)` Drops some embedding channels which regularises better than normal dropout for sequences 
- `Bidirectional(LSTM(32))` Reads the sequence both forwards and backwards, better for capturing context 
- `Dense(16, relu)` Adds a non-linear transformation 
- `Dropout(0.4)` High dropout to slow overfitting 
- `Dense(softmax)` Outputs a probability distribution for the rating classes 

In [ ]:
EMBEDDING_DIM = 64

model = Sequential([
    Embedding(input_dim=MAX_VOCAB,
              output_dim=EMBEDDING_DIM,
              input_length=MAX_LEN),
    SpatialDropout1D(0.3),
    Bidirectional(LSTM(32, dropout=0.3, recurrent_dropout=0.3)),
    Dense(16, activation='relu'),
    Dropout(0.4),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.build(input_shape=(None, MAX_LEN))
model.summary()

## Training the Model

- **Class weights** helps the model pay more attention to underrepresented ratings
- **EarlyStopping** on validation loss with patience=5. This stops the training when the model doesn't get any meaningfully better

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_val, y_val),
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1
)

## Evaluation on Test Set

- **Accuracy** overall fraction of correct predictions
- **Classification report** precision, recall, and F1 per class. The Macro avg F1 is the
  most meaningful metric here because it treats all classes equally
- **Confusion matrix** shows where the model makes mistakes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['loss'],        label='Train Loss')
axes[0].plot(history.history['val_loss'],    label='Val Loss')
axes[0].set_title('Loss over Epochs')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend()

axes[1].plot(history.history['accuracy'],    label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'],label='Val Accuracy')
axes[1].set_title('Accuracy over Epochs')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].legend()

plt.tight_layout()
plt.show()

**Loss and accuracy**
As we see on the graph, around the third epoch is where the model starts to overfit. 
Learning the training data (getting better training accuracy), but not improving its test accuracy

In [ ]:
y_score = model.predict(X_test_final)
y_pred = np.argmax(y_score, axis=1)

print(f"Test Accuracy: {accuracy_score(y_test_final, y_pred):.4f}\n")

target_names = [f"Rating {i}" for i in range(1, NUM_CLASSES + 1)]
print(classification_report(y_test_final, y_pred, target_names=target_names, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test_final, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(1, NUM_CLASSES + 1),
            yticklabels=range(1, NUM_CLASSES + 1))
plt.title('Confusion Matrix - Evaluated on Original Data Only')
plt.xlabel('Predicted Rating')
plt.ylabel('True Rating')
plt.tight_layout()
plt.show()

## Precision-Recall Curves

Because the dataset is imbalanced, precision-recall curves give a clearer picture than ROC curves of
how well the model identifies each rating class
- A **one-vs-rest** strategy is used: for each class the model's predicted probability for that class
is treated as the score, and all other classes are treated as negatives
- The **Average precision** summarises the area under each curve.

In [ ]:
y_test_bin = label_binarize(y_test_final, classes=range(NUM_CLASSES))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig, ax = plt.subplots(figsize=(8, 6))

for i in range(NUM_CLASSES):
    precision, recall, _ = precision_recall_curve(y_test_bin[:, i], y_score[:, i])
    ap = average_precision_score(y_test_bin[:, i], y_score[:, i])
    ax.plot(recall, precision, color=colors[i], lw=2,
            label=f'Rating {i + 1} (AP = {ap:.2f})')

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves (One-vs-Rest)')
ax.legend(loc='lower left')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
plt.tight_layout()
plt.show()

## Summary 

### Results
- Class, F1-score
- Rating 1, 0.53
- Rating 2, 0.11
- Rating 3, 0.21
- Rating 4, 0.19
- Rating 5, 0.69
- accuracy, 0.50
- macro avg, 0.35
- weighted avg, 0.52

### Model takeaways
Our LSTM achieves around 50% test accuracy and a macro F1 of around 0.35. While these numbers might not be the best, the model does however demonstrate learning:

- **Ratings 1 and 5** are predicted well (F1 of 0.53 and 0.69). The model has
  learned to distinguish clearly positive from clearly negative sentiment
- **Ratings 2, 3, and 4** are poorly predicted (F1 of 0.11-0.21). These classes have far fewer
  training samples (61–90 test samples vs 229–471 for ratings 1 and 5) and their review texts
  are often ambiguous

### Why the performance could be limited

- **Severe class imbalance:** Rating 5 has almost half of the samples while rating 2 has less than 10%. Even with
   class weights, the model has far fewer examples to learn what a 2-star review looks like
- **Short, uninformative reviews:** Many reviews are just 1–3 words.
There is simply not enough text to distinguish a 4-star rating from a 5-star one
- **Noisy labels:** Some 5-star reviews contain complaints and some 1-star 
reviews mention positive features. The text-rating is not perfect 
- **Small dataset:** ~6200 reviews is small for training an LSTM with a 20 000-word
   vocabulary. The embedding layer alone has 1.28M parameters which is far more than the number of
   training samples
- **Overfitting:** Despite regularisation (dropout, smaller architecture), the model begins
   overfitting around epoch 3, as we can see by the divergence between training and
   validation loss

### Steps taken to improve performance

- **Class weights** (`compute_class_weight('balanced')`) 
- **Bidirectional LSTM** allows the model to capture context from both directions
- **Model size reduction** We reduced the model size throughout (LSTM 64→32, Dense 32→16) and increased dropout (0.2→0.3/0.4)
to try to combat overfitting on the small dataset
- **Text preprocessing** lowercasing, URL/email removal, removing some stop words and keeping others that
  carry sentiment
- **Text augmentation** to add more data the model can train on. This is done to combat the heavily unbalanced dataset

### Possible further improvements

- **Reduce to 3 classes** (Negative=1–2, Neutral=3, Positive=4–5), this could significantly
   boost performance by grouping ambiguous middle ratings and addressing the imbalance in the dataset
- **Transformer-based models** Using a more complex transformer like BERT could possibly improve the model performance
